# Label propagation on a kNN graph

A scanner's readings form two noisy rings around the origin, plus a small cluster far from
both of a kind nobody has labeled yet. One point of each ring is labeled. You will build the
k-nearest-neighbor graph, run label propagation on it, and solve for its fixed point
directly, and in both cases report the points that no label can reach instead of guessing a
class for them.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 250 points on a ring of radius 1, 250 on a ring of radius 2.5, and 40 points
# in a far cluster. y_partial holds -1 for every unlabeled point.
rng = np.random.default_rng(12)
turn = rng.uniform(0, 2 * np.pi, 500)
size = np.r_[np.full(250, 1.0), np.full(250, 2.5)] + rng.normal(0, 0.08, 500)
z = size * np.exp(1j * turn)                       # each ring point as a complex number
X = np.r_[np.c_[z.real, z.imag], rng.normal(7.0, 0.3, (40, 2))]
ring = np.r_[np.zeros(250, int), np.ones(250, int), np.full(40, -1)]   # -1 marks the far cluster
y_partial = np.full(len(X), -1)
y_partial[[0, 250]] = [0, 1]
print(f"{len(X)} points, labeled: {np.flatnonzero(y_partial >= 0)}")

## Part 1 (coding task): The kNN graph

Write `knn_graph(X, k)` for an $n \times d$ array of points. Join points $i$ and $j$ when
$j$ is one of the $k$ nearest other points of $i$, or $i$ is one of the $k$ nearest other
points of $j$. Return the $n \times n$ weight matrix with 1 for an edge and 0 elsewhere,
including the diagonal.

Distances here never tie, so you need not worry about ties.

In [ ]:
def knn_graph(X, k):
    """Symmetric 0/1 adjacency of the k-nearest-neighbor graph, zero diagonal."""
    X = np.asarray(X, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
line = np.array([[0.0], [1.0], [3.0], [7.0], [8.5]])
W1 = knn_graph(line, 1)
assert W1.shape == (5, 5), f"expected a 5 x 5 matrix, got shape {W1.shape}"
assert np.all(np.diag(W1) == 0), "a point must not be its own neighbor: the diagonal should be 0"
assert np.array_equal(W1, W1.T), "the graph must be symmetric: join i and j when either picks the other"
edges1 = {(int(i), int(j)) for i, j in zip(*np.nonzero(W1)) if i < j}
assert edges1 == {(0, 1), (1, 2), (3, 4)}, f"k = 1 edges {sorted(edges1)}: point 2's nearest is point 1, so 1-2 is an edge even though point 1 picks point 0"
W2 = knn_graph(line, 2)
edges2 = {(int(i), int(j)) for i, j in zip(*np.nonzero(W2)) if i < j}
assert edges2 == {(0, 1), (0, 2), (1, 2), (2, 3), (2, 4), (3, 4)}, f"k = 2 edges {sorted(edges2)}"
Wx = knn_graph(X, 10)
assert np.array_equal(Wx, Wx.T) and set(np.unique(Wx)) == {0.0, 1.0}, "on the rings: expected a symmetric 0/1 matrix"
assert Wx.sum(axis=1).min() >= 10, "every point should have at least k neighbors"
print("Part 1 passed")

## Part 2 (coding task): Propagate, and say when no label arrives

Write `propagate(W, y_partial, n_classes, n_iter=1000)`. Normalize each row of `W` to sum
to 1, start from one-hot rows for the labeled points and zeros elsewhere, and repeat
`n_iter` times: every point takes the weighted average of its neighbors' rows, then the
labeled rows are reset to their one-hot labels. Return `(labels, F)`, where `F` is the final
$n \times$ `n_classes` array and `labels` is each row's best class, or `-1` for a point that
no labeled point can reach.

Every row of `W` has at least one edge.

In [ ]:
def propagate(W, y_partial, n_classes, n_iter=1000):
    """Iterate-and-clamp label propagation. Unreached points get the label -1."""
    # TODO
    raise NotImplementedError

In [ ]:
# Seven points: 0 and 4 are labeled, 5 and 6 form a piece of their own.
G = np.zeros((7, 7))
for i, j, w in [(0, 1, 2.0), (1, 2, 1.0), (2, 3, 1.0), (1, 3, 1.0), (3, 4, 1.0), (5, 6, 1.0)]:
    G[i, j] = G[j, i] = w
yG = np.array([0, -1, -1, -1, 1, -1, -1])
labels, F = propagate(G, yG, 2)
assert F.shape == (7, 2), f"F should have one row per point and one column per class, got {F.shape}"
assert np.allclose(F[[0, 4]], [[1, 0], [0, 1]]), "the labeled rows must be reset to their one-hot labels every sweep"
assert np.allclose(F[1:4, 0], [10 / 13, 8 / 13, 6 / 13], atol=1e-6), f"P(class 0) at points 1-3 is {F[1:4, 0]}, expected 10/13, 8/13, 6/13: normalize each row of W, not each column"
assert np.allclose(F[5:], 0), "points 5 and 6 are not connected to any label, so their rows should stay 0"
assert labels.tolist() == [0, 0, 0, 1, 1, -1, -1], f"labels {labels.tolist()}: expected [0, 0, 0, 1, 1, -1, -1], and unreached points get -1, not the first class"
print("Part 2 passed")

## Part 3 (coding task): The fixed point in one solve

Write `closed_form(W, y_partial, n_classes)`, returning `(labels, F)` like Part 2 but with
the fixed point solved for directly. For the reached unlabeled points $U$ and the labeled
points $L$, the fixed point of Part 2 satisfies $F_U = T_{UU} F_U + T_{UL} Y_L$. Give
unreached points a zero row and the label `-1`.

The check then runs both methods on the rings with $k = 10$.

In [ ]:
def closed_form(W, y_partial, n_classes):
    """Label propagation's fixed point by a linear solve. Unreached points get -1."""
    # TODO
    raise NotImplementedError

In [ ]:
try:
    labels, F = closed_form(G, yG, 2)
except np.linalg.LinAlgError as err:
    raise AssertionError(f"the solve failed ({err}): I - T_UU is singular when U includes points no label can reach") from None
assert np.allclose(F[1:4, 0], [10 / 13, 8 / 13, 6 / 13], atol=1e-9), f"P(class 0) at points 1-3 is {F[1:4, 0]}, expected 10/13, 8/13, 6/13"
assert np.allclose(F[5:], 0) and labels.tolist() == [0, 0, 0, 1, 1, -1, -1], f"labels {labels.tolist()}: points 5 and 6 should get a zero row and -1"

W = knn_graph(X, 10)
lab_iter, _ = propagate(W, y_partial, 2)
lab_solve, F = closed_form(W, y_partial, 2)
assert np.array_equal(lab_iter, lab_solve), "the solve and the iteration should give the same labels on the rings"
on_rings = ring >= 0
acc = np.mean(lab_solve[on_rings] == ring[on_rings])
nearest = np.argmin(((X[:, None, :] - X[None, [0, 250], :]) ** 2).sum(-1), axis=1)
print(f"ring points labeled correctly: {acc:.3f}   (nearest labeled point: {np.mean(nearest[on_rings] == ring[on_rings]):.3f})")
print(f"far cluster: {np.sum(lab_solve[~on_rings] == -1)} of {np.sum(~on_rings)} points left at -1")
assert acc > 0.99 and np.all(lab_solve[~on_rings] == -1), "expected the rings labeled almost perfectly and the far cluster left at -1"
print("Part 3 passed")

## Solution

### Part 1: The kNN graph

In [ ]:
def knn_graph(X, k):
    """Symmetric 0/1 adjacency of the k-nearest-neighbor graph, zero diagonal."""
    X = np.asarray(X, dtype=float)
    sq = (X ** 2).sum(axis=1)
    d2 = sq[:, None] + sq[None, :] - 2 * X @ X.T     # squared distances, all pairs
    d2[np.arange(len(X)), np.arange(len(X))] = np.inf   # a point is not its own neighbor
    nearest = np.argsort(d2, axis=1)[:, :k]
    W = np.zeros((len(X), len(X)))
    W[np.repeat(np.arange(len(X)), k), nearest.ravel()] = 1.0
    return np.maximum(W, W.T)                        # keep an edge chosen from either end

**Why it works.** Masking the diagonal with infinity keeps a point from choosing itself, which
it otherwise always would at distance 0. The k-nearest relation is not symmetric: in the
check, point 2 picks point 1 but point 1 picks point 0. Taking the elementwise maximum of
`W` and its transpose keeps an edge chosen from either end, which is the usual kNN graph.
Keeping only edges chosen from both ends gives the mutual kNN graph, which can leave points
with no neighbors at all.

### Part 2: Propagate, and say when no label arrives

In [ ]:
def propagate(W, y_partial, n_classes, n_iter=1000):
    """Iterate-and-clamp label propagation. Unreached points get the label -1."""
    T = W / W.sum(axis=1, keepdims=True)             # row i: point i's similarity as fractions
    labeled = y_partial >= 0
    Y = np.zeros((len(W), n_classes))
    Y[labeled, y_partial[labeled]] = 1.0
    F = Y.copy()
    for _ in range(n_iter):
        F = T @ F
        F[labeled] = Y[labeled]
    labels = F.argmax(axis=1)
    labels[F.sum(axis=1) == 0] = -1                  # no label ever arrived
    return labels, F

**Why it works.** Dividing by the row sums makes row $i$ of `T` a probability distribution over
$i$'s neighbors, so `T @ F` averages the neighbors' label rows. Dividing by the column
sums instead gives a different operator whenever degrees differ, and the check's point 1,
with weighted degree 4, catches it. A point in a piece of the graph with no labeled point only ever
averages zeros, so its row stays exactly 0, and `argmax` of a zero row returns class 0. The
row sum is the honest test: it is 1 for every reached point and 0 for the rest.

### Part 3: The fixed point in one solve

In [ ]:
def closed_form(W, y_partial, n_classes):
    """Label propagation's fixed point by a linear solve. Unreached points get -1."""
    n = len(W)
    labeled = np.flatnonzero(y_partial >= 0)
    reached = y_partial >= 0
    while True:                                      # grow the reached set one hop at a time
        grown = reached | (W[:, reached].sum(axis=1) > 0)
        if grown.sum() == reached.sum():
            break
        reached = grown
    U = np.flatnonzero(reached & (y_partial < 0))
    T = W / W.sum(axis=1, keepdims=True)
    Y = np.zeros((n, n_classes))
    Y[labeled, y_partial[labeled]] = 1.0
    F = Y.copy()
    F[U] = np.linalg.solve(np.eye(len(U)) - T[np.ix_(U, U)], T[np.ix_(U, labeled)] @ Y[labeled])
    labels = np.full(n, -1)
    labels[reached] = F[reached].argmax(axis=1)
    return labels, F

**Why it works.** The reached set grows by one hop per pass: a point joins when it has an
edge to a point already in the set, and it stops growing when a pass adds nothing. Solving
over every unlabeled point fails, because a piece of the graph with no label gives
$I - T_{UU}$ a block whose rows sum to zero, a singular matrix. Restricted to reached points,
every point has a path to a labeled point, so within a bounded number of steps some of its
weight leaks to a label: $T_{UU}^n$ tends to zero, its spectral radius is below 1, and $I -
T_{UU}$ is invertible. On the rings each label spreads around its own ring and never crosses the gap,
while the nearest labeled point, the supervised answer with two labels, gets about a third
of the ring points wrong.